In [ ]:
import pandas as pd, sqlalchemy as sqla, numpy as np
import decouple
from sqlalchemy import String, Integer
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', 1000)

mis_file_path = decouple.config("MIS_FILE_PATH")

mis_db = sqla.create_engine(decouple.config("MIS_DB"))
red_db = sqla.create_engine(decouple.config("RED_DB"))

print(mis_db)
print(red_db)

In [ ]:
# truncate existing
with red_db.connect() as conn:
    conn.execute(sqla.text("TRUNCATE TABLE redreport2026.overall_schedules;"))
    conn.execute(sqla.text("TRUNCATE TABLE redreport2026.overall_account_categories;"))
    conn.execute(sqla.text("TRUNCATE TABLE redreport2026.overall_universe_accounts;"))

In [ ]:
# Schedule

# Get dimensions
store_details = pd.read_sql_table(table_name="store_details", con=mis_db, schema="ref")

# Get approved schedules
red_sched = pd.read_excel(f"{mis_file_path}MIS Data\\RED Alert\\RED Bulk Upload.xlsx", sheet_name="Approved Schedule")
red_sched = red_sched[["Day", "Employee Code", "Account Name", "Store Code"]]
red_sched = red_sched.drop_duplicates()

red_sched = pd.merge(
    red_sched, 
    store_details[["account_name", "store_code", "bpc_store_code"]],
    left_on=["Account Name", "Store Code"],
    right_on=["account_name", "store_code"],
    how="left"
)

red_sched["schedule_id"] = range(1, len(red_sched) + 1, 1)
red_sched = red_sched[["schedule_id", "Day", "Employee Code", "bpc_store_code"]]

red_sched = red_sched.rename(columns={
    "Day": "day_of_the_week",
    "Employee Code": "emp_code"
})

# Load to database
red_sched.to_sql(name="overall_schedules", con=red_db, schema="redreport2026", if_exists="append", index=False, dtype={
    "schedule_id": Integer,
    "day_of_the_week": String(255),
    "emp_code": String(500),
    "bpc_store_code": String(1000)
})

display(red_sched.shape)
display(red_sched.head(5))
display(red_sched.tail(5))

In [ ]:
# Promos

store_details = pd.read_sql_table(table_name="store_details", con=mis_db, schema="ref")
store_details["account_code"] = store_details["bpc_store_code"].str.extract(r"BPC-(\D+)-\d+")
store_details["promo_name"] = "N/A"
store_details = store_details[["account_code", "account_name", "promo_name"]]
store_details = store_details.drop_duplicates()

promos = pd.read_excel(f"{mis_file_path}MIS Data\\RED Alert\\RED Bulk Upload.xlsx", sheet_name="Promos")

# Append N/A and promos
red_promos = pd.concat([promos, store_details], ignore_index=True)

red_promos = red_promos.sort_values(["account_name", "promo_name"])
red_promos["id"] = range(1, len(red_promos) + 1, 1)

red_promos = red_promos[["id", "account_code", "account_name", "promo_name"]]
red_promos = red_promos.reset_index(drop=True)

# Load to database
red_promos.to_sql(name="overall_account_categories", con=red_db, schema="redreport2026", if_exists="append", index=False, dtype={
    "id": Integer,
    "account_code":	String(255),		
    # "account_name":	String(),		
    "promo_name":	String(10000)
})

display(red_promos.shape)
display(red_promos.head(5))
display(red_promos.tail(5))

In [ ]:
# Stores / Accounts

# Get dimensions
store_details = pd.read_sql_table(table_name="store_details", con=mis_db, schema="ref")
account_details = pd.read_sql_table(table_name="account_details", con=mis_db, schema="ref")
print(store_details.shape)
store_details = pd.merge(
        store_details,
        account_details[["account_name", "account_chain"]],
        on=["account_name"],
        how="left"
    )
store_details["account_code"] = store_details["bpc_store_code"].str.extract(r"BPC-(\D+)-\d+")
print(store_details.shape)

# Get approved schedules
red_stores = pd.read_excel(f"{mis_file_path}MIS Data\\RED Alert\\RED Bulk Upload.xlsx", sheet_name="Approved Schedule", dtype="str")
red_stores = red_stores[["Account Name", "Store Code", "TOS", "Employee Code"]]
red_stores = red_stores.drop_duplicates(subset=["Account Name", "Store Code", "TOS", "Employee Code"])

store_details = pd.merge(
    store_details,
    red_stores,
    left_on=["account_name", "store_code"],
    right_on=["Account Name", "Store Code"],
    how="left"
)
print(store_details.shape)

store_details = store_details[["bpc_store_code","store_code","account_code","account_chain","store_name","city","province","region","bpc_region", "TOS", "Employee Code"]]

store_details = store_details.rename(columns={
    "account_chain": "account_chain_name",
    "store_name": "account_store_name",
    "TOS": "account_tas",
    "Employee Code": "account_merchandiser"
})



# store_details
store_details.to_csv(r"V:\MIS Data\RED Alert\new_red_sched.csv", index=False, encoding="utf-8")

In [ ]:
# Load to database
store_details.to_sql(name="overall_universe_accounts", con=red_db, schema="redreport2026", if_exists="append", index=False, dtype={
    "bpc_store_code": String(255),
    "store_code": String(255),
    "account_code": String(255),
    "account_chain_name": String(255),
    "account_store_name": String(255),
    "city": String(255),
    "province": String(255),
    "region": String(255),
    "bpc_region": String(255),
    "account_tas": String(255),
    "account_merchandiser": String(1000)
})

display(red_stores.shape)
display(red_stores.head(5))
display(red_stores.tail(5))